# TSUM CryptoBERT LoRA 파인튜닝

**목표**: `ElKulako/cryptobert` 모델을 수집한 크립토 뉴스 데이터로 파인튜닝해서
`bearish / neutral / bullish` 분류 성능을 높인다.

## 순서
1. GPU 확인 및 패키지 설치
2. 데이터 업로드 및 확인
3. 데이터셋 전처리
4. LoRA 파인튜닝
5. 평가 (accuracy, F1)
6. 모델 저장 및 다운로드

> **실행 전**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

---
## Step 1. GPU 확인 및 패키지 설치

In [ ]:
import subprocess, sys

# GPU 확인
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('GPU 없음 — 런타임 유형을 T4 GPU로 변경하세요')

In [ ]:
%%capture
!pip install -q \
    transformers>=4.41.0 \
    datasets>=2.19.0 \
    peft>=0.11.0 \
    accelerate>=0.30.0 \
    scikit-learn>=1.4.0 \
    pandas numpy

print('설치 완료')

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, f1_score, classification_report

warnings.filterwarnings('ignore')
set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')

---
## Step 2. 데이터 업로드

로컬에서 `python training/collect_data.py` 실행 후 생성된
`data/crypto_news_labeled.csv` 파일을 업로드하세요.

In [ ]:
from google.colab import files

print('crypto_news_labeled.csv 파일을 선택하세요')
uploaded = files.upload()
csv_path = list(uploaded.keys())[0]
print(f'업로드됨: {csv_path}')

In [ ]:
df = pd.read_csv(csv_path)
print(f'총 {len(df):,}개 샘플')
print(df['label'].value_counts())
df.head()

---
## Step 3. 데이터 전처리

In [ ]:
# ── 설정 (필요 시 변경) ──────────────────────────────────────────────────────
BASE_MODEL   = 'ElKulako/cryptobert'
OUTPUT_DIR   = 'finetuned_cryptobert'   # 저장 경로
MAX_LENGTH   = 192
TEST_SIZE    = 0.1
EPOCHS       = 3
BATCH_SIZE   = 16     # T4 VRAM 16GB 기준, OOM 시 8로 낮추세요
LR           = 2e-4
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.1

LABELS    = ['bearish', 'neutral', 'bullish']
LABEL2ID  = {l: i for i, l in enumerate(LABELS)}
ID2LABEL  = {i: l for l, i in LABEL2ID.items()}

In [ ]:
# 텍스트 정제 + 라벨 정규화
df = df.dropna(subset=['text'])
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() >= 10].drop_duplicates('text').reset_index(drop=True)

if 'label' not in df.columns and 'price_change_24h' in df.columns:
    df['label'] = df['price_change_24h'].apply(
        lambda x: 'bullish' if x >= 1.5 else ('bearish' if x <= -1.5 else 'neutral')
    )

df['label'] = df['label'].str.lower().map(
    lambda x: x if x in LABEL2ID else 'neutral'
)
df['label_id'] = df['label'].map(LABEL2ID)

print(f'정제 후: {len(df):,}개')
print(df['label'].value_counts())

# 클래스 불균형 확인
counts = df['label'].value_counts()
if counts.max() / counts.min() > 5:
    print('\n⚠️  클래스 불균형이 심합니다. 데이터를 더 수집하거나 --days를 늘려보세요.')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

dataset = Dataset.from_pandas(df[['text', 'label_id']])
split = dataset.train_test_split(test_size=TEST_SIZE, seed=42, stratify_by_column='label_id')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

tokenized = split.map(tokenize, batched=True, remove_columns=['text'])
tokenized = tokenized.rename_column('label_id', 'labels')
tokenized.set_format('torch')

print(f'Train: {len(tokenized["train"]):,}  Test: {len(tokenized["test"]):,}')

---
## Step 4. LoRA 파인튜닝

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['query', 'value'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=(DEVICE == 'cuda'),
    logging_steps=50,
    report_to='none',
    dataloader_pin_memory=(DEVICE == 'cuda'),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

---
## Step 5. 평가

In [ ]:
metrics = trainer.evaluate()
print(f"\n최종 결과:")
print(f"  accuracy : {metrics['eval_accuracy']:.4f}")
print(f"  macro_f1 : {metrics['eval_macro_f1']:.4f}")

In [ ]:
# 클래스별 상세 리포트
preds_out = trainer.predict(tokenized['test'])
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = preds_out.label_ids

print(classification_report(
    [ID2LABEL[i] for i in y_true],
    [ID2LABEL[i] for i in y_pred],
    labels=LABELS,
))

---
## Step 6. 모델 저장 및 다운로드

다운로드된 `finetuned_cryptobert.zip`을 압축 해제한 뒤
**`tsum/models/finetuned_cryptobert/`** 폴더 안에 넣으세요.

```
tsum/
└── models/
    └── finetuned_cryptobert/   ← 여기
        ├── config.json
        ├── tokenizer_config.json
        ├── adapter_config.json
        ├── adapter_model.safetensors
        └── ...
```

Render 배포 시엔 모델 파일이 크므로 **Hugging Face Hub에 업로드**하거나
`MODEL_PATH` 환경변수로 경로를 지정하는 걸 권장합니다.

In [ ]:
import shutil

# 최종 모델 저장
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'모델 저장 완료: {OUTPUT_DIR}/')

# 파일 목록 확인
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    size = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:40s}  {size:.1f} MB')

In [ ]:
# zip으로 압축 후 다운로드
zip_path = f'{OUTPUT_DIR}.zip'
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f'압축 완료: {zip_path}  ({Path(zip_path).stat().st_size/1024/1024:.1f} MB)')

files.download(zip_path)
print('다운로드 시작됩니다...')

---
## (선택) Hugging Face Hub에 업로드

Render 서버에서 모델을 바로 불러오고 싶다면 HF Hub에 올리세요.
업로드 후 `.env`에 `MODEL_PATH=<your-hf-id>/finetuned_cryptobert` 설정.

In [ ]:
# HF_TOKEN은 https://huggingface.co/settings/tokens 에서 발급
# HF_REPO = 'your-hf-username/finetuned_cryptobert'

# from huggingface_hub import login
# login(token='hf_...')
# trainer.model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print(f'업로드 완료: https://huggingface.co/{HF_REPO}')